In [1]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling
from pyproj import CRS
from matplotlib import pyplot as plt
from tqdm import tqdm
import os
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
POP_DIR = RESULTS / "population-rasters"
POP_DIR.mkdir(parents=True, exist_ok=True)

### population in 2020

In [5]:
population_2020_ = rio.open(DATA / 'population' / 'ppp_2020_1km_Aggregated.tif')

In [ ]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
country_raster_10km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-10km.tif')
country_raster_50km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-50km.tif')

In [7]:
population_2020 = population_2020_.read(1, masked=True).filled(np.nan)

In [ ]:
population_2020_1km  = np.zeros(shape=country_raster_1km_.shape, dtype=np.float32)
population_2020_10km = np.zeros(shape=country_raster_10km_.shape, dtype=np.float32)
population_2020_50km = np.zeros(shape=country_raster_50km_.shape, dtype=np.float32)

In [ ]:
reproject(
    source=population_2020,
    destination=population_2020_1km,
    src_transform=population_2020_.transform,
    src_crs=population_2020_.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.sum,
    src_nodata=np.nan,
    dst_nodata=np.nan
)

In [ ]:
reproject(
    source=population_2020,
    destination=population_2020_10km,
    src_transform=population_2020_.transform,
    src_crs=population_2020_.crs,
    dst_transform=country_raster_10km_.transform,
    dst_crs=country_raster_10km_.crs,
    resampling=Resampling.sum,
    src_nodata=np.nan,
    dst_nodata=np.nan
)

(array([[nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.]],
       shape=(1694, 3386), dtype=float32),
 Affine(10000.0, 0.0, -16930000.0,
        0.0, -10000.0, 8470000.0))

In [8]:
reproject(
    source=population_2020,
    destination=population_2020_50km,
    src_transform=population_2020_.transform,
    src_crs=population_2020_.crs,
    dst_transform=country_raster_50km_.transform,
    dst_crs=country_raster_50km_.crs,
    resampling=Resampling.sum,
    src_nodata=np.nan,
    dst_nodata=np.nan
)

(array([[nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.]],
       shape=(340, 678), dtype=float32),
 Affine(50000.0, 0.0, -16950000.0,
        0.0, -50000.0, 8500000.0))

In [ ]:
with rio.open(
    POP_DIR / 'population-2020-1km.tif',
    "w",
    driver="GTiff",
    height=country_raster_1km_.shape[0],
    width=country_raster_1km_.shape[1],
    count=1,
    dtype=population_2020_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(population_2020_1km, 1)

In [ ]:
with rio.open(
    POP_DIR / 'population-2020-10km.tif',
    "w",
    driver="GTiff",
    height=country_raster_10km_.shape[0],
    width=country_raster_10km_.shape[1],
    count=1,
    dtype=population_2020_10km.dtype,
    crs=country_raster_10km_.crs,
    transform=country_raster_10km_.transform,
    compress='lzw'
) as dst:
    dst.write(population_2020_10km, 1)

In [ ]:
with rio.open(
    POP_DIR / 'population-2020-50km.tif',
    "w",
    driver="GTiff",
    height=country_raster_50km_.shape[0],
    width=country_raster_50km_.shape[1],
    count=1,
    dtype=population_2020_50km.dtype,
    crs=country_raster_50km_.crs,
    transform=country_raster_50km_.transform,
    compress='lzw'
) as dst:
    dst.write(population_2020_50km, 1)

### Future population

In [ ]:
def reproject_future_population(year, scenario='SSP2', res='1km'):
    country_raster_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    input_path = DATA / 'population' / scenario / f'{scenario}_{year}.tif'

    src = rio.open(input_path)
    population_grid = src.read(1, masked=True).filled(np.nan)
    population_grid_eck4_ = np.zeros(shape=country_raster_.shape, dtype=np.float32)

    reproject(
        source=population_grid,
        destination=population_grid_eck4_,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=country_raster_.transform,
        dst_crs=country_raster_.crs,
        resampling=Resampling.sum,
        src_nodata=np.nan,
        dst_nodata=np.nan
    )

    # print(np.sum(np.where(population_grid_eck4_ > 0, population_grid_eck4_, 0)) / np.nansum(population_grid))

    os.makedirs(POP_DIR / scenario, exist_ok=True)
    output_path = POP_DIR / scenario / f'population-{year}-{scenario}-{res}.tif'

    with rio.open(
        output_path, 
        "w",
        driver="GTiff",
        height=country_raster_.shape[0],
        width=country_raster_.shape[1],
        count=1,
        dtype=population_grid_eck4_.dtype,
        crs=country_raster_.crs,
        transform=country_raster_.transform,
        compress='lzw'
    ) as dst:
        dst.write(population_grid_eck4_, 1)

In [ ]:
for scenario in tqdm(['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']):
    for year in tqdm(range(2025, 2055, 5)):
        reproject_future_population(year, scenario=scenario, res='1km')

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
for scenario in tqdm(['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']):
    for year in tqdm(range(2025, 2055, 5)):
        reproject_future_population(year, scenario=scenario, res='10km')

In [ ]:
for scenario in tqdm(['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']):
    for year in tqdm(range(2025, 2055, 5)):
        reproject_future_population(year, scenario=scenario, res='50km')